# 05 · θ + z vs single-z representations: links to drug response, phenotypes and patient data

Question: does splitting a VABAM model's representation into θ and z map onto drug response, published circadian phenotypes and patient-level expression better than a single mixed z?

| Representation | Content |
|---|---|
| `VABAM_<model>_z` | VABAM z only |
| `VABAM_<model>_theta` | θ response: amplitude change and shape preservation of the generated signal when θ_L is swept from 0.05 to 0.95 (8 dims) |
| `VABAM_<model>_ztheta` | z + θ response |
| `BaseVAE_CIRC_30_s*`, `TCVAE_*`, `ConVAE_*` | single-z VAEs from the official benchmarks |
| `PCA30_raw` | linear floor without training |
| classical | cosinor and autocorrelation features (reference) |

- The inferred θ is constant across inputs, so θ is represented by the model's response to θ as a control.
- The downstream analysis (`src/exp02/analyze.py`) is identical for every representation.
- Runs on CPU.

In [ ]:
import os, sys
from pathlib import Path
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
sys.dont_write_bytecode = True
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run.py").exists() and (p / "configs").exists())
sys.path.insert(0, str(ROOT))
import pandas as pd
from src import repmap as rm
pd.set_option("display.width", 220, "display.max_columns", 30)
%matplotlib inline

MODEL = "SKZFC_CIRC_1_30_200"     # or a sweep run, e.g. "s3/s3_MULTI_FC12_Z10_s0"
N_PERM, N_RANDOM = 200, 300        # permutations / random signatures

## 1. Public data (≈1.4 GB, downloaded once)

In [ ]:
from src.exp02 import fetch_external
fetch_external.main()

## 2. Representations
θ response, z and z+θ are written to `results/exp02/codes/`, together with the PCA floor.

In [ ]:
rm.write_codes(MODEL)
rm.pca_codes()

## 3. Single-z baselines (trained in a terminal)

```
python run_exp02.py baseline --type BaseVAE --seed 0 --gpu 0
python run_exp02.py baseline --type BaseVAE --seed 1 --gpu 0
python run_exp02.py baseline --type BaseVAE --seed 2 --gpu 0
python run_exp02.py baseline --type TCVAE  --seed 0 --gpu 0
python run_exp02.py baseline --type ConVAE --seed 0 --gpu 0
```

## 4. Analysis
Representations without results are analysed; output in `results/exp02/<rep>/`.

In [ ]:
rm.analyze_missing(N_PERM, N_RANDOM)

## 5. Comparison

In [ ]:
T = rm.table()
display(T.round(3))
rm.plot_table(T);

## Notes

- θ contribution: `_z` vs `_ztheta`. Disentangled vs mixed: `_ztheta` vs `BaseVAE` (difference must exceed the spread across BaseVAE seeds).
- A learned representation is useful only if it improves on `PCA30_raw`. The published phenotypes are defined from classical metrics, so the classical row is expected to score well.
- GDSC: counts must exceed the 95th percentile of the cell-line permutation null.
- With 13–15 cell lines, small differences are not interpretable. The θ response is a deterministic function of z (θ is constant), i.e. a re-expression of z along the amplitude-control direction rather than new information.